## Goal

The goal of this notebook is to scrape data needed for a book recommendation project. The way the project is going to work is it takes an input of Kdramas you have enjoyed and recommend books that are similar to those Kdramas. It's more of a since you liked this drama, you might like this book. It is for those who enjoy reading books and watching Kdramas like me <3.

#### Data Needed
- Kdrama data (title, genre, rating, synopsis, etc.)
- Book data (title, genre, rating, synopsis, etc.)

#### Sources
- Kdrama data: https://mydramalist.com/k-dramas
- Book data: https://www.goodreads.com (albeit obtained from kaggle via https://www.kaggle.com/datasets/ishanrealstate/goodreads-cleaned-dataset)

For the Kdrama data, I couldn't get all I needed in a place so I scraped titles off KdramaList using selenium and then used the mydramalistAPI to get the rest of the data. As for book data, a more comprehensive dataset was available on kaggle but if you wish to scrape, amazon is a good place to look into. Another good place is using hardcover API.

https://www.amazon.com/s?i=stripbooks&s=relevancerank&Adv-Srch-Books-Submit.x=19&Adv-Srch-Books-Submit.y=9&unfiltered=1&ref=sr_adv_b

https://docs.hardcover.app/api/getting-started/

### Scraping Kdrama Names and Links

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import time
import random

In [3]:
def scrape_pages(start_page, end_page):
    chrome_path = "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome"

    service = Service()  # Selenium will try Selenium Manager again
    options = Options()
    options.binary_location = chrome_path

    driver = webdriver.Chrome(service=service, options=options)

    for page in range(start_page, end_page + 1):
        url = f"https://mydramalist.com/search?adv=titles&ty=68&co=3&re=2000,2026&so=top&page={page}"
        print(f"Scraping page {page}...")
        
        driver.get(url)

        # wait for JS + Cloudflare
        time.sleep(random.uniform(4, 7))

        titles = driver.find_elements(By.CSS_SELECTOR, "h6.text-primary.title a")
        if not titles:
            print(f"No titles found on page {page}")

        with open("title.txt", "a", encoding="utf-8") as f:
            for title in titles:
                text = title.text.strip()
                link = title.get_attribute('href')
                if text and link:
                    f.write(f"{text} | {link}\n")

        time.sleep(random.uniform(2, 5))

    driver.quit()
    print("Batch complete.\n")

In [4]:
total_pages = 250
batch_size = 10

In [8]:
for start in range(215, total_pages+1, batch_size):
    end = min(start+batch_size-1,total_pages)
    scrape_pages(start,end)
    time.sleep(random.uniform(10,20))

Scraping page 215...
Scraping page 216...
Scraping page 217...
Scraping page 218...
Scraping page 219...
Scraping page 220...
Scraping page 221...
Scraping page 222...
Scraping page 223...
Scraping page 224...
Batch complete.

Scraping page 225...
Scraping page 226...
Scraping page 227...
Scraping page 228...
Scraping page 229...
Scraping page 230...
Scraping page 231...
Scraping page 232...
Scraping page 233...
Scraping page 234...
Batch complete.

Scraping page 235...
Scraping page 236...
Scraping page 237...
Scraping page 238...
Scraping page 239...
Scraping page 240...
Scraping page 241...
Scraping page 242...
Scraping page 243...
Scraping page 244...
Batch complete.

Scraping page 245...
Scraping page 246...
No titles found on page 246
Scraping page 247...
No titles found on page 247
Scraping page 248...
No titles found on page 248
Scraping page 249...
No titles found on page 249
Scraping page 250...
No titles found on page 250
Batch complete.



### Scraping full kdrama data from my dramalist API

In [2]:
import pandas as pd
import requests
import json
import time

In [3]:
movies = pd.read_csv('title.txt',sep='|',names=["Title","Links"])

In [4]:
movies['slug'] = movies['Links'].apply(lambda x: x.split('/')[-1].strip()) #extracting slug from link

In [5]:
movies

,Title,Links,slug
0,When Life Gives You Tangerines,https://mydramalist.com/735043-life,735043-life
1,Twinkling Watermelon,https://mydramalist.com/739603-sparkling-wate...,739603-sparkling-watermelon
2,Move to Heaven,https://mydramalist.com/49231-move-to-heaven,49231-move-to-heaven
3,Weak Hero Class 1,https://mydramalist.com/702267-weak-hero,702267-weak-hero
4,Alchemy of Souls,https://mydramalist.com/52939-can-this-person...,52939-can-this-person-be-translated
...,...,...,...
4888,Fancafe,https://mydramalist.com/56253-fancafe,56253-fancafe
4889,On the Turf,https://mydramalist.com/704693-on-the-turf,704693-on-the-turf
4890,KBS Drama Special 2012: Pandemonium,https://mydramalist.com/16625-drama-special-2...,16625-drama-special-2012-pandemonium
4891,Between,https://mydramalist.com/68555-between,68555-between


In [6]:
movie_slugs = list(movies.slug)

In [7]:
def write_to_txt(movie_details,output_file,count):
    """helper function: function to write movie details extracted to csv"""
    with open(output_file,'a',encoding='utf-8') as f:
        for md in movie_details:
            f.write(json.dumps(md) + '\n')
        print(f"{count} movie details writen to {output_file}")

In [8]:
def get_movie_details(slugs, output_file = 'Movie_File.jsonl', batch_size = 50):
    """This function gets movie details from dramalist API and uses write_to_txt function to write to a file"""
    session = requests.Session()
    movie_details = []
    for i, slug in enumerate(slugs, 1):
        link = f"https://my-drama-list-api-ten.vercel.app/api/id/{slug}"
        for attempt in range(3):    
            try:
                response = session.get(link, timeout=10)
                response.raise_for_status()
                movie_detail = response.json()
                movie_details.append(movie_detail)
                break
            except requests.exceptions.RequestException:
                print(f"Attempt {attempt + 1} failed for slug {slug}")
                if attempt == 2:
                    print(f"Failed to get movie details for slug {slug}")
                else:
                    time.sleep(0.5)
        if i % batch_size==0:
            write_to_txt(movie_details, output_file,count=i)
            movie_details = []
        time.sleep(0.2)
    if movie_details:
        write_to_txt(movie_details, output_file,count=i)
    print("All done")
        

In [9]:
get_movie_details(slugs=movie_slugs[450:])

50 movie details writen to Movie_File.jsonl
100 movie details writen to Movie_File.jsonl
150 movie details writen to Movie_File.jsonl
200 movie details writen to Movie_File.jsonl
250 movie details writen to Movie_File.jsonl
Attempt 1 failed for slug 33412-love-affairs-in-the-afternoon
Attempt 2 failed for slug 33412-love-affairs-in-the-afternoon
Attempt 3 failed for slug 33412-love-affairs-in-the-afternoon
Failed to get movie details for slug 33412-love-affairs-in-the-afternoon
300 movie details writen to Movie_File.jsonl
350 movie details writen to Movie_File.jsonl
400 movie details writen to Movie_File.jsonl
450 movie details writen to Movie_File.jsonl
500 movie details writen to Movie_File.jsonl
550 movie details writen to Movie_File.jsonl
600 movie details writen to Movie_File.jsonl
650 movie details writen to Movie_File.jsonl
700 movie details writen to Movie_File.jsonl
750 movie details writen to Movie_File.jsonl
800 movie details writen to Movie_File.jsonl
850 movie details writ

In [10]:
data = pd.read_json('Movie_File.jsonl',lines = True)

In [11]:
data.to_csv('raw_kdrama.csv')

### Scraping Book Details

In [ ]:
import pandas as pd
import sys

In [ ]:
book_data = pd.read_parquet('Goodreads Books/books_clean.parquet')

In [ ]:
book_data.to_csv('raw_books_data.csv', index=False)

In [ ]:
#dividing into chunks of 90000 for the sake of github upload
start = 0
end = 40000
for i in range(0,50):
    book_data[start:end].to_csv(f'books_data_{i}.csv', index=False)
    start = end
    end += 40000

In [ ]:
book_data